# Native FlashNext training CPU gate

No GPU or full checkpoint allocation. Pinned experimental Transformers GGUF fork, native vision processor, packed-weight tiny activation gradients. AMD-specific optimized kernels are not installed. Apache2.0 notices for Qwen, Hugging Face and the reviewed recipe remain applicable. No performance claim.

In [ ]:
BASE_LOCK_SHA256='4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02'
ARCHIVE_SHA256='10e5df7fbe82f82f4cbf95e923e465b900184e36fa9877fbafdf752e6b549961'
EXPECTED_INPUT_MANIFEST={'files': {'16k/train.jsonl': {'sha256': 'fd73b3802e886bd4dc6c443ac24388f3c491f7720b6213ded89e4bc5889d71c5', 'bytes': 7031316}, '16k/validation.jsonl': {'sha256': 'c8527dde162787d496e842a6a4684cf692e9490211b8ce520176ade02c8ea452', 'bytes': 2815735}, '32k/train.jsonl': {'sha256': 'd3e8c271f7fc67c3c5ff72019946ff5a46bf4f90f0bbc973b08da58c44c19efe', 'bytes': 18380809}, '32k/validation.jsonl': {'sha256': 'c40c876828c0338036fdde844e1d47f7d66f575dfc22e5e9721a33dc39489d74', 'bytes': 4696845}, 'wheels/accelerate-1.15.0-py3-none-any.whl': {'sha256': '97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec', 'bytes': 394295}, 'wheels/fla_core-0.6.0-py3-none-any.whl': {'sha256': 'b0708b70e0b119cd58a0a5146ea56cb175cef6ed969d8ac60267e3e077ce3ebd', 'bytes': 1085221}, 'wheels/flash_linear_attention-0.6.0-py3-none-any.whl': {'sha256': '6dec9735f8454523c13553622e98aff30152dde6586b4ac7d88010fbb7a06b5f', 'bytes': 403549}, 'wheels/peft-0.21.2-py3-none-any.whl': {'sha256': '106ab6077ff72c54d21577f9af5970e34bac7582cb14209f7b2b511e322a4eae', 'bytes': 835427}, 'arc3_training_collator.py': {'sha256': 'e27f3881e0b83ab3f3cdc4da216158ed5dd69e6807891e5ea925702766170858', 'bytes': 2460}, 'split.json': {'sha256': 'ce36006c553278e8282b9e8ca05bfa0b6f6b39a9ffc831d03dd9419e760e5525', 'bytes': 1233}, 'base-requirements.lock': {'sha256': '4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02', 'bytes': 4187}}, 'purpose': 'Private offline ARC3 training/evaluation inputs. No pretrained weights, credentials, hidden games or test-game examples.', 'attribution': 'Agent/prompt source Daniel Franzen, Jeroen Cottaar and Tufa Labs, Apache2.0. Our own recorded Qwen agent outputs. Upstream dependency licenses remain inside wheels. Preserve applicable original notices; no blanket relicensing of third-party material.'}
from pathlib import Path
from hashlib import sha256
import json,os,subprocess,sys,zipfile,urllib.request
work=Path('/kaggle/working');bundle=work/'arc3-training-dependencies';bundle.mkdir(exist_ok=True)
roots=[Path('/kaggle/input/datasets/dfranzen/pennyroyal-v253'),Path('/kaggle/input/pennyroyal-v253')]
root=next(p for p in roots if p.is_dir());wheels=root/'wheels';lock=root/'requirements.lock'
assert sha256(lock.read_bytes()).hexdigest()==BASE_LOCK_SHA256,'Base lock changed'
environment=dict(os.environ,UV_OFFLINE='1',UV_PYTHON_DOWNLOADS='never',PYTHONNOUSERSITE='1',
    HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',USE_TF='0',USE_TORCHVISION='0')
environment.pop('PYTHONPATH',None)
venv=Path('/tmp/arc3-training-cpu-env');uv=Path('/tmp/arc3-training-uv');python=venv/'bin/python'
uv_wheels=list(wheels.glob('uv-*.whl'));assert len(uv_wheels)==1
with zipfile.ZipFile(uv_wheels[0]) as archive:
    names=[n for n in archive.namelist() if Path(n).name=='uv' and not n.endswith('/')];assert len(names)==1
    uv.write_bytes(archive.read(names[0]));uv.chmod(0o755)
def run(arguments):
    result=subprocess.run(list(map(str,arguments)),env=environment,text=True,capture_output=True)
    print(result.stdout[-3500:],flush=True)
    if result.returncode:print(result.stderr[-7000:],flush=True);raise RuntimeError('Isolated CPU command failed')
run([uv,'venv','--python',sys.executable,venv])
run([uv,'pip','install','--python',python,'--no-index','--find-links',wheels,'-r',lock])
import tarfile
input_roots=[Path('/kaggle/input/datasets/kunalsinghh25/arc3-training-pilot-inputs-v1'),Path('/kaggle/input/arc3-training-pilot-inputs-v1')]
assets=work/'training-assets';assets.mkdir(exist_ok=True)
archives=[root/'training-pilot-inputs.tar.gz' for root in input_roots if (root/'training-pilot-inputs.tar.gz').is_file()]
if archives:
    assert len(archives)==1
    archive=archives[0]
    assert sha256(archive.read_bytes()).hexdigest()==ARCHIVE_SHA256,'Private input archive changed'
    tar=tarfile.open(archive,'r:gz')
    manifest=json.loads(tar.extractfile('manifest.json').read())
    def read_asset(name):
        member=tar.getmember(name)
        if not member.isfile():raise ValueError('Invalid archive member')
        return tar.extractfile(member).read()
    input_asset_mode='archive-sha256-and-file-sha256'
else:
    # Kaggle automatically expanded this uploaded tar.gz. The API file listing
    # verifies this exact prefix; validate every original manifest/file hash.
    roots=[root/'training-pilot-inputs' for root in input_roots if (root/'training-pilot-inputs/manifest.json').is_file()]
    if len(roots)!=1:raise ValueError('Pinned expanded assets are not uniquely mounted: '+str(roots))
    source_root=roots[0].resolve();manifest=json.loads((source_root/'manifest.json').read_text())
    def read_asset(name):
        source=(source_root/name).resolve()
        if not source.is_relative_to(source_root) or not source.is_file():raise ValueError('Invalid expanded asset path')
        return source.read_bytes()
    input_asset_mode='kaggle-expanded-exact-manifest-and-file-sha256'
assert manifest==EXPECTED_INPUT_MANIFEST
try:
    for name,record in manifest['files'].items():
        destination=(assets/name).resolve()
        if not destination.is_relative_to(assets.resolve()):raise ValueError('Asset escapes output root')
        content=read_asset(name)
        assert len(content)==record['bytes'] and sha256(content).hexdigest()==record['sha256'],name
        destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(content)
finally:
    if archives:tar.close()
(assets/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('TRAINING_ASSET_INTEGRITY',input_asset_mode,flush=True)

import shutil
for wheel in (assets/'wheels').glob('*.whl'):shutil.copyfile(wheel,bundle/wheel.name)
run([uv,'pip','install','--python',python,'--no-index','--no-deps',*sorted(bundle.glob('*.whl'))])
environment['UV_OFFLINE']='0'
def installed_distributions():
    code='import importlib.metadata as m,json; print(json.dumps({d.metadata["Name"].lower().replace("_","-"):d.version for d in m.distributions()}))'
    return json.loads(subprocess.check_output([str(python),'-c',code],env=environment,text=True))
base_distributions=installed_distributions()
archive=work/'transformers-source.tar.gz'
with urllib.request.urlopen('https://codeload.github.com/woct0rdho/transformers/tar.gz/e6564cfd7efd3affc3b6f3d65738f80b7ad8474f',timeout=90) as response:archive.write_bytes(response.read())
assert sha256(archive.read_bytes()).hexdigest()=='c3c892aadd9043fac06dfe0d1cdd0c175b31e15f905f4a2e4d3bbc609855ca91','Pinned library source changed'
sources=work/'transformers-source';sources.mkdir(exist_ok=True)
with tarfile.open(archive,'r:gz') as handle:handle.extractall(sources,filter='data')
project=next(p for p in sources.iterdir() if p.is_dir())
run([uv,'build','--wheel','--out-dir',bundle,project])
run([uv,'pip','uninstall','--python',python,'sglang'])
constraints=work/'torch-constraint.txt';constraints.write_text('torch==2.13.0+cu130\n')
run([uv,'pip','install','--python',python,'--constraint',constraints,*sorted(bundle.glob('transformers-*.whl'))])
run([uv,'pip','check','--python',python])
installed=installed_distributions()
changed=[name+'=='+version for name,version in installed.items() if name!='transformers' and base_distributions.get(name)!=version]
if changed:run([sys.executable,'-m','pip','download','--no-deps','--only-binary=:all:','--dest',bundle,*changed])
(bundle/'manifest.json').write_text(json.dumps(dict(source_archive_sha256='c3c892aadd9043fac06dfe0d1cdd0c175b31e15f905f4a2e4d3bbc609855ca91',
    installed_distributions=installed,changed_requirements=changed,
    files={p.name:sha256(p.read_bytes()).hexdigest() for p in bundle.glob('*.whl')}),indent=2)+'\n')
probe=work/'flashnext_native_probe.py';probe.write_text("from pathlib import Path\nfrom hashlib import sha256\nfrom collections import Counter\nimport json,torch,transformers,peft,accelerate\nfrom transformers import AutoConfig,AutoModelForImageTextToText,AutoProcessor\nfrom accelerate import init_empty_weights\nfrom transformers.integrations.gguf.reader import GgufHeader,read_gguf_metadata\nfrom transformers.integrations.gguf.gguf_quantized_parameter import GgufQuantizedParameter\nfrom transformers.integrations.gguf.modules import GgufLinear\nfrom transformers.integrations.gguf.moe import _project_expert\nfrom peft import LoraConfig,get_peft_model\ntorch.set_num_threads(4)\nwork=Path('/kaggle/working');assets=work/'training-assets'\nbase=Path('/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1')\nquant=Path('/kaggle/input/models/buianhtruc/strata-qwen3-8-flash-next-gsq-rco-iq3-s/gguf/default/1')\nreport=dict(phase='checking',cuda_available=torch.cuda.is_available(),torch=torch.__version__,transformers=transformers.__version__,peft=peft.__version__,errors=[])\ndef save():\n    (work/'flashnext-training-cpu-probe.json').write_text(json.dumps(report,indent=2)+'\\n')\nsave()\ntry:\n    assert not torch.cuda.is_available(),'CPU metadata must not allocate a GPU'\n    assert torch.__version__=='2.13.0+cu130','Existing runtime Torch must be retained'\n    files=sorted(quant.glob('*.gguf'));assert len(files)==2\n    headers=[GgufHeader.from_file(str(path)) for path in files]\n    names=[entry.name for header in headers for entry in header.tensors]\n    assert len(names)==len(set(names)) and 'per_layer_token_embd.weight' in names\n    report['gguf_headers']=[dict(file=path.name,bytes=path.stat().st_size,architecture=header.architecture,\n        tensors=len(header.tensors),types=dict(Counter(entry.ggml_type for entry in header.tensors)),\n        header_sha256=sha256(json.dumps([entry._asdict() for entry in header.tensors],sort_keys=True).encode()).hexdigest()) for path,header in zip(files,headers)]\n    report['split_loading_required']=True\n    config=AutoConfig.from_pretrained(base,local_files_only=True,trust_remote_code=False)\n    with init_empty_weights():model=AutoModelForImageTextToText.from_config(config,dtype=torch.bfloat16)\n    targets=[name for name,module in model.named_modules() if isinstance(module,torch.nn.Linear)\n        and 'language_model.layers.' in name and '.indexer.' not in name\n        and name.rsplit('.',1)[-1] in ['q_proj','k_proj','v_proj','o_proj','in_proj_qkv','in_proj_z','out_proj','gate_proj','up_proj','down_proj']]\n    assert len(targets)==300,('Unexpected native language projection inventory',len(targets))\n    assert all(p.device.type=='meta' for p in model.parameters())\n    report.update(model_class=type(model).__name__,native_parameters=sum(p.numel() for p in model.parameters()),\n        ordinary_lora_targets=targets,vision_parameters=sum(p.numel() for name,p in model.named_parameters() if '.visual.' in name))\n    del model\n    # Real CPU activation gradients through persistent quantized bytes. The\n    # independently known Q4_0 block encodes16 ones followed by16 zeros.\n    payload=torch.zeros(32,18,dtype=torch.uint8);payload[:,:2]=torch.tensor([0,60],dtype=torch.uint8);payload[:,2:]=0x89\n    weight=GgufQuantizedParameter(payload,quant_type=2,logical_shape=(32,32))\n    layer=GgufLinear(32,32,bias=False,dtype=torch.float32,compute_dtype=torch.float32);layer.weight=weight\n    expected=torch.cat([torch.ones(32,16),torch.zeros(32,16)],dim=1)\n    x=torch.randn(3,32,requires_grad=True);gradient=torch.randn(3,32)\n    actual=layer(x);actual.backward(gradient)\n    assert torch.allclose(actual,torch.nn.functional.linear(x,expected),atol=1e-6,rtol=1e-6)\n    assert torch.allclose(x.grad,gradient@expected,atol=1e-6,rtol=1e-6)\n    expert_payload=payload.unsqueeze(0).repeat(4,1,1)\n    expert_weight=GgufQuantizedParameter(expert_payload,quant_type=2,logical_shape=(4,32,32))\n    selected=torch.tensor([2]);z=torch.randn(3,32,requires_grad=True)\n    projected=_project_expert(z,expert_weight,selected,torch.float32,'eager');projected.backward(gradient)\n    assert torch.allclose(z.grad,gradient@expected,atol=1e-6,rtol=1e-6)\n    class Toy(torch.nn.Module):\n        def __init__(self):super().__init__();self.projection=layer\n        def forward(self,value):return self.projection(value)\n    adapted=get_peft_model(Toy(),LoraConfig(r=4,lora_alpha=8,target_modules=['projection'],bias='none'))\n    parameters=[p for p in adapted.parameters() if p.requires_grad]\n    loss=adapted(torch.randn(3,32)).square().mean();loss.backward()\n    assert parameters and all('lora_' in name for name,p in adapted.named_parameters() if p.requires_grad)\n    assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum()>0 for p in parameters)\n    report['tiny_quantized_activation_gradients']=dict(passed=True,linear_and_selected_expert=True,peft_adapter_gradients=True,\n        boundary='Actual tiny CPU Q4_0 math only; not CUDA grouped-MM, full IQ3_S training or ARC3 quality.')\n    processor=AutoProcessor.from_pretrained(base,local_files_only=True,trust_remote_code=False)\n    namespace={};exec((assets/'arc3_training_collator.py').read_text(),namespace);collate=namespace['collate_example']\n    prepared={'train':[],'validation':[]};rejected=[]\n    allocation=json.loads((assets/'split.json').read_text())['games']\n    for split in prepared:\n        rows=[json.loads(line) for line in (assets/'32k'/f'{split}.jsonl').read_text().splitlines()]\n        for row in rows:\n            assert allocation[row['metadata']['game_id']]==split,'Frozen whole-game training allocation changed'\n            kwargs=row['template_kwargs'];render=processor.tokenizer.apply_chat_template\n            prefix=render(row['messages'][:-1],tools=row['tools'],tokenize=False,add_generation_prompt=True,**kwargs)\n            text=render(row['messages'],tools=row['tools'],tokenize=False,add_generation_prompt=False,**kwargs)\n            if not text.startswith(prefix):raise ValueError('Native FlashNext assistant boundary differs')\n            row['assistant_start_char']=len(prefix);row['rendered_text_sha256']=sha256(text.encode()).hexdigest()\n            try:batch,info=collate(row,processor,max_tokens=16384)\n            except ValueError as error:\n                if str(error)=='Training context limit exceeded':\n                    rejected.append(dict(example=row['example_id'],reason='Actual native context exceeds16384'));continue\n                raise\n            row['supervised_text_tokens']=info['supervised_tokens'];row['native_flashnext_tensor_check']=info\n            prepared[split].append(row);del batch\n    for split,rows in prepared.items():\n        (work/f'flashnext-{split}.jsonl').write_text(''.join(json.dumps(row,separators=(',',':'))+'\\n' for row in rows))\n    assert prepared['train'] and prepared['validation']\n    report.update(phase='complete',passed=True,dataset_counts={k:len(v) for k,v in prepared.items()},rejected_contexts=rejected,\n        dataset_sha256={k:sha256((work/f'flashnext-{k}.jsonl').read_bytes()).hexdigest() for k in prepared},\n        boundary='CPU native multimodal processing and meta-model inventory plus tiny quantized activation/LoRA gradients. No full model weights, CUDA training, split-loader/vision-weight integration, saved adapter serving or gameplay improvement verified.')\n    save();print(json.dumps({k:v for k,v in report.items() if k not in ['ordinary_lora_targets','rejected_contexts']},indent=2))\nexcept BaseException as error:\n    report.update(phase='failed',passed=False,error_type=type(error).__name__,error=str(error)[:1000]);save();raise\n")
run([python,probe])
